In [1]:
import os
import numpy as np
import faiss
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("all good")

all good


In [2]:
def load_docs(folder="docs"):
    texts = []
    for fname in os.listdir(folder):
        if fname.endswith(".txt"):
            with open(os.path.join(folder, fname), encoding="utf-8") as f:
                texts.append(f.read())
    return texts

documents = load_docs()
print(f"Loaded {len(documents)} documents")

Loaded 3 documents


In [3]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_text(doc))

print(f"{len(all_chunks)} chunks")

134 chunks


In [5]:
def embed(texts):
    resp = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([d.embedding for d in resp.data], dtype="float32")

def embed_batched(texts, batch_size=100):
    out = []
    for i in range(0, len(texts), batch_size):
        out.append(embed(texts[i:i + batch_size]))
    return np.vstack(out)

chunk_vectors = embed_batched(all_chunks)
print(chunk_vectors.shape)

(134, 1536)


In [6]:
dim = chunk_vectors.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(chunk_vectors)
print(f"Index holds {index.ntotal} vectors")

Index holds 134 vectors


In [7]:
def ask(question, k=3, show_chunks=False):
    q_vec = embed([question])
    distances, ids = index.search(q_vec, k)
    retrieved = [all_chunks[i] for i in ids[0]]

    if show_chunks:
        for n, c in enumerate(retrieved):
            print(f"--- chunk {n} (dist {distances[0][n]:.2f}) ---")
            print(c[:200], "\n")

    context = "\n---\n".join(retrieved)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer using only the provided context. If the context does not contain the answer, say you don't know."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )
    return resp.choices[0].message.content

print(ask("What is the difference between an image and a container in Docker?", show_chunks=True))

--- chunk 0 (dist 0.80) ---
ects are images, containers, and services.
- A Docker container is a standardized, encapsulated environment that runs applications. A container is managed using the Docker API or CLI.
- A Docker image 

--- chunk 1 (dist 0.92) ---
ent process that manages Docker containers and handles container objects. The daemon listens for requests that are sent via the Docker Engine API. The Docker client program, calleddocker, provides a c 

--- chunk 2 (dist 0.97) ---
This is similar to how containers on Linux work. 
- "8 surprising facts about real Docker adoption". Datadog. June 2018. Retrieved September 4, 2019.
-  Gupta, Devender (October 13, 2022). "How to Ins 

A Docker image is a read-only template used to build containers, while a Docker container is a standardized, encapsulated environment that runs applications based on that image.


In [8]:
# Experiment 1: question NOT in your docs — does it say "I don't know" or make something up?
print(ask("Who won the 2022 World Cup?"))

# Experiment 2: retrieval depth — same question, k=1 vs k=8, compare answer quality
print(ask("How does BERT handle input text?", k=1))
print(ask("How does BERT handle input text?", k=8))

I don't know.
BERT handles input text by being a deeply bidirectional, unsupervised language representation that is pre-trained using only a plain text corpus. It takes into account the context for each occurrence of a given word, in contrast to context-free models that generate a single word embedding representation for each word.
BERT handles input text by using a deeply bidirectional approach to learn contextual representations. It takes into account the context for each occurrence of a word rather than generating a single word embedding for each word in isolation. BERT is trained using masked token prediction and next sentence prediction, allowing it to learn the relationships between tokens in their context. It applies a self-attention mechanism to gather information from both the left and right sides of the text during training, which contributes to its deep understanding of context.
